# 배송 지연 1~7일 — 일별 평점 변화 분석

**목적**: 예정 배송일 기준 1~7일 지연 그룹의 일별 평점 변화를 시각화하고,  
각 지연일 기준으로 정상 배송 가정 시 전체 평점이 얼마나 개선되는지 추정한다.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

ROOT = Path('../../').resolve()
df_raw = pd.read_csv(ROOT / 'data/processed/olist_order_item_level.csv')
df_raw['order_delivered_customer_date'] = pd.to_datetime(df_raw['order_delivered_customer_date'], errors='coerce')
df_raw['order_estimated_delivery_date'] = pd.to_datetime(df_raw['order_estimated_delivery_date'], errors='coerce')

d = df_raw[
    (df_raw['order_status'] == 'delivered') &
    (df_raw['anomaly_flag'] == 0) &
    df_raw['review_score_mean'].notna()
].copy()
d['delay_days'] = (d['order_delivered_customer_date'] - d['order_estimated_delivery_date']).dt.days

total        = len(d)
overall_avg  = d['review_score_mean'].mean()
score_day0   = d[d['delay_days'] == 0]['review_score_mean'].mean()

# 0~7일 일별 통계
stats = (
    d[d['delay_days'].between(0, 7)]
    .groupby('delay_days')['review_score_mean']
    .agg(avg='mean', n='count', std='std')
    .reset_index()
)
stats['se']            = stats['std'] / np.sqrt(stats['n'])
stats['drop_vs_prev']  = stats['avg'].diff().round(4)
stats['drop_vs_day0']  = (stats['avg'] - score_day0).round(4)

# 반사실: 해당 delay_days 초과 주문이 정시(day=0) 평점을 받았다면 전체 평균
cf_avgs = []
for day in range(0, 8):
    mask = d['delay_days'] > day
    n_fix = mask.sum()
    cf = (d['review_score_mean'].sum() - d.loc[mask, 'review_score_mean'].sum() + n_fix * score_day0) / total
    cf_avgs.append(cf)
stats['cf_overall_avg'] = cf_avgs

# 1점/5점 비율
for score, col in [(1, 'pct_1'), (5, 'pct_5')]:
    rate = (
        d[d['delay_days'].between(0,7)]
        .assign(is_target=lambda x: x['review_score_mean'].round() == score)
        .groupby('delay_days')['is_target'].mean() * 100
    )
    stats = stats.merge(rate.rename(col).reset_index(), on='delay_days')

print(f'전체 평균 평점: {overall_avg:.4f}점  (분석 대상: {total:,}건)')
print(f'정시(0일) 평점: {score_day0:.4f}점\n')
print(stats[['delay_days','avg','n','drop_vs_prev','drop_vs_day0','pct_1','pct_5','cf_overall_avg']]
      .to_string(index=False))

전체 평균 평점: 4.0806점  (분석 대상: 107,786건)
정시(0일) 평점: 3.9863점

 delay_days      avg    n  drop_vs_prev  drop_vs_day0     pct_1     pct_5  cf_overall_avg
        0.0 3.986335 1427           NaN        0.0000  9.810792 49.894884        4.193961
        1.0 3.678879  928       -0.3075       -0.3075 15.193966 40.948276        4.191313
        2.0 3.140411  584       -0.5385       -0.8459 29.109589 29.280822        4.186730
        3.0 2.570922  564       -0.5695       -1.4154 46.453901 23.226950        4.179324
        4.0 2.518908  476       -0.0520       -1.4674 47.689076 21.848739        4.172843
        5.0 2.143878  490       -0.3750       -1.8425 57.142857 13.673469        4.164468
        6.0 1.816934  437       -0.3269       -2.1694 68.192220 10.297483        4.155672
        7.0 1.885553  533        0.0686       -2.1008 63.227017  9.380863        4.145284


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

days   = stats['delay_days'].values
avgs   = stats['avg'].values
ns     = stats['n'].values
ses    = stats['se'].values

# ── [1] 일별 평균 평점 꺾은선 + 95% 신뢰구간
ax = axes[0][0]
ax.fill_between(days, avgs - 1.96*ses, avgs + 1.96*ses, alpha=0.15, color='#d7191c')
ax.plot(days, avgs, color='#d7191c', lw=2.8, marker='o', ms=9, zorder=5)

for x, y, n in zip(days, avgs, ns):
    ax.annotate(f'{y:.2f}점\n(n={n:,})',
                xy=(x, y), xytext=(0, 12), textcoords='offset points',
                ha='center', fontsize=8.5, color='#d7191c', fontweight='bold')

# 전날 대비 하락폭 화살표
for i in range(1, len(days)):
    drop = avgs[i] - avgs[i-1]
    mx   = (days[i-1] + days[i]) / 2
    my   = (avgs[i-1] + avgs[i])  / 2
    ax.text(mx, my - 0.13, f'{drop:+.2f}', ha='center', fontsize=8,
            color='#555555', style='italic')

ax.axhline(overall_avg, color='gray', lw=1.2, ls='--', alpha=0.7,
           label=f'현재 전체 평균 ({overall_avg:.2f}점)')
ax.axhline(3.0, color='#fdae61', lw=1.0, ls=':', alpha=0.8, label='3.0점 기준선')

ax.set_xticks(days)
ax.set_xticklabels([f'{int(d)}일' for d in days], fontsize=10)
ax.set_ylabel('평균 리뷰 평점', fontsize=11)
ax.set_ylim(1.2, 5.0)
ax.set_title('지연일별 평균 평점\n(숫자: 전일 대비 변화)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# ── [2] 1점 vs 5점 비율 변화
ax = axes[0][1]
ax.plot(days, stats['pct_5'], color='#1a9641', lw=2.5, marker='s', ms=7, label='5점 비율')
ax.plot(days, stats['pct_1'], color='#d7191c', lw=2.5, marker='o', ms=7, label='1점 비율')
ax.fill_between(days, stats['pct_1'], stats['pct_5'], alpha=0.08, color='gray')

for x, y1, y5 in zip(days, stats['pct_1'], stats['pct_5']):
    ax.text(x, y5 + 1.0, f'{y5:.0f}%', ha='center', fontsize=8, color='#1a9641')
    ax.text(x, y1 - 2.5, f'{y1:.0f}%', ha='center', fontsize=8, color='#d7191c')

# 1점 > 5점 역전 지점
cross_idx = next((i for i in range(len(days)) if stats['pct_1'].iloc[i] >= stats['pct_5'].iloc[i]), None)
if cross_idx:
    ax.axvline(days[cross_idx], color='black', lw=1.5, ls='--', alpha=0.7)
    ax.text(days[cross_idx] + 0.1, 55, f'역전\n({int(days[cross_idx])}일)', fontsize=8.5)

ax.axhline(50, color='gray', lw=0.8, ls=':', alpha=0.5)
ax.set_xticks(days)
ax.set_xticklabels([f'{int(d)}일' for d in days], fontsize=10)
ax.set_ylabel('비율 (%)', fontsize=11)
ax.set_ylim(0, 75)
ax.set_title('1점 vs 5점 비율 변화\n(두 선이 교차하는 지점 = 역전일)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# ── [3] 전일 대비 평점 하락폭
ax = axes[1][0]
drops      = stats['drop_vs_prev'].fillna(0).values
bar_colors = ['#d7191c' if v < -0.3 else '#fdae61' if v < -0.1 else '#a6d96a' for v in drops]
bars = ax.bar(days, drops, color=bar_colors, edgecolor='white', linewidth=0.8, width=0.6)

for bar, val in zip(bars, drops):
    if val != 0:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() - 0.015,
                f'{val:.3f}', ha='center', va='top', fontsize=9,
                color='white', fontweight='bold')

ax.axhline(0, color='black', lw=0.8)
ax.axhline(-0.3, color='#d7191c', lw=1.0, ls=':', alpha=0.6, label='-0.3점 기준선')
ax.set_xticks(days)
ax.set_xticklabels([f'{int(d)}일' for d in days], fontsize=10)
ax.set_ylabel('전일 대비 평점 변화', fontsize=11)
ax.set_title('전일 대비 평점 하락폭\n(음수 = 하락, 빨강 = 0.3점 이상 하락)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# ── [4] 반사실: 해당 delay 초과 주문 해결 시 전체 평균
ax = axes[1][1]
ax2 = ax.twinx()

fix_counts = [d[d['delay_days'] > day].shape[0] for day in days]
ax2.bar(days, fix_counts, color='lightgray', alpha=0.4, width=0.6, label='해결 대상 건수')
ax2.set_ylabel('해결 대상 주문 수', color='gray', fontsize=10)
ax2.tick_params(axis='y', labelcolor='gray')
ax2.set_ylim(0, max(fix_counts) * 3)

ax.plot(days, stats['cf_overall_avg'], color='#2c7bb6', lw=2.8, marker='D', ms=8, zorder=5)
ax.axhline(overall_avg, color='gray', lw=1.2, ls='--', alpha=0.7, label=f'현재 {overall_avg:.4f}점')

for x, y in zip(days, stats['cf_overall_avg']):
    gain = y - overall_avg
    ax.text(x, y + 0.001, f'{y:.4f}\n(+{gain:.4f})',
            ha='center', va='bottom', fontsize=7.5, color='#2c7bb6', fontweight='bold')

ax.set_xticks(days)
ax.set_xticklabels([f'{int(d)}일 초과\n해결' for d in days], fontsize=8)
ax.set_ylabel('반사실 전체 평균 평점', fontsize=11)
ax.set_ylim(overall_avg - 0.005, max(stats['cf_overall_avg']) + 0.015)
ax.set_title(f'각 지연일 초과 주문 해결 시\n전체 평균 평점 (현재 {overall_avg:.4f}점)',
             fontsize=12, fontweight='bold')
lines1, lbs1 = ax.get_legend_handles_labels()
lines2, lbs2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, lbs1 + lbs2, fontsize=8.5)

plt.suptitle('배송 지연 1~7일 일별 평점 변화 분석', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 요약 테이블
summary = stats[['delay_days','avg','n','drop_vs_prev','pct_1','pct_5','cf_overall_avg']].copy()
summary.columns = ['지연일','평균평점','주문수','전일대비','1점비율(%)','5점비율(%)','반사실전체평균']
summary['지연일'] = summary['지연일'].apply(lambda x: f'{int(x)}일')
summary['전일대비'] = summary['전일대비'].apply(lambda x: f'{x:+.3f}' if pd.notna(x) and x!=0 else '-')
summary['1점비율(%)'] = summary['1점비율(%)'].round(1)
summary['5점비율(%)'] = summary['5점비율(%)'].round(1)
summary['반사실전체평균'] = summary['반사실전체평균'].round(4)

print(f'현재 전체 평균: {overall_avg:.4f}점')
print()
print(summary.to_string(index=False))
print()
print('▶ 핵심 인사이트')
print(f'  · 가장 큰 일별 하락: 2→3일 ({stats["drop_vs_prev"].min():.3f}점), 1→2일 ({stats["drop_vs_prev"].iloc[2]:.3f}점)')
print(f'  · 1점이 5점을 역전하는 시점: {int(stats.loc[stats["pct_1"]>=stats["pct_5"],"delay_days"].iloc[0])}일')
print(f'  · 1일 지연만 막아도 전체 평균: {stats["cf_overall_avg"].iloc[1]:.4f}점 (+{stats["cf_overall_avg"].iloc[1]-overall_avg:.4f}점)')
print(f'  · 3일 지연까지 막으면: {stats["cf_overall_avg"].iloc[3]:.4f}점 (+{stats["cf_overall_avg"].iloc[3]-overall_avg:.4f}점)')